# 05 - Risk ontology and state population rates

This stage maps CFPB product, issue, and sub-issue text to five project categories: **Fraud/Identity Theft**, **Billing Dispute**, **Credit Reporting Error**, **Debt Collection Misconduct**, and **Other**. Rules are explicit and editable in `src/ontology_features.py`; each row also receives a short `risk_category_rule` explaining the matched phrase. The mapping is a heuristic feature, not a CFPB label or a fraud finding.

It also joins a user-provided state population CSV to complaint counts and calculates `complaints_per_100k = state_complaint_count / state_population * 100000`. Raw counts tend to be larger in more populous states simply because more people live there. Per-capita rates put states on a common population denominator, making state comparisons fairer. They still do not adjust for reporting access, complaint propensity, customer mix, or other confounders.

In [ ]:
# Find the project root and import the reusable ontology and population helpers.
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.ontology_features import (
    add_ontology_and_population_features,
    map_risk_category,
    process_complaints_csv,
)

# Preview how a few CFPB-like taxonomy combinations map to the custom ontology.
taxonomy_examples = pd.DataFrame(
    [
        ("Credit card", "Unauthorized transaction", "Identity theft"),
        ("Credit card", "Billing disputes", "Incorrect charge"),
        ("Credit reporting", "Incorrect information on your report", "Account status"),
        ("Debt collection", "Communication tactics", "Frequent or repeated calls"),
        ("Mortgage", "Application processing delay", ""),
    ],
    columns=["Product", "Issue", "Sub-issue"],
)
mapped_examples = taxonomy_examples.apply(
    lambda row: map_risk_category(row["Product"], row["Issue"], row["Sub-issue"]),
    axis=1,
)
taxonomy_examples["risk_category"] = mapped_examples.map(lambda result: result[0])
taxonomy_examples["risk_category_rule"] = mapped_examples.map(lambda result: result[1])
display(taxonomy_examples)

## Mapping rules and precedence

The mapper normalizes case and punctuation, then checks phrases across Product, Issue, and Sub-issue. It applies categories in this order: fraud/identity-theft indicators first, then debt-collection misconduct, credit-reporting error, billing dispute, and finally Other. This makes an explicit unauthorized/fraud phrase take precedence over a broad product such as a credit card. When no phrase matches, the `Debt collection` and `Credit reporting` product families provide a specific fallback; unrelated products are not forced into a risk category. Review the `risk_category_rule` field and amend the lists when the CFPB taxonomy snapshot includes relevant new wording.

## Join the Census population CSV

Provide a state-level population CSV with one population estimate per state. The script accepts postal abbreviations, full state names, or Census FIPS codes for the join key. Pass the actual Census header names if they differ from the default columns `state` and `population`; for example, use `NAME` and `POPESTIMATE2024`. Non-state aggregate rows such as `United States` are ignored, but duplicate recognized states or invalid/non-positive populations fail clearly. Complaint states without a matching population remain null and are listed in a summary JSON.

In [ ]:
# Set these file paths to the CFPB export and the Census file you provide.
complaints_csv = project_root / "data" / "raw" / "complaints.csv"
population_csv = project_root / "data" / "raw" / "state_population.csv"
enriched_csv = project_root / "data" / "processed" / "complaints_ontology_population.csv"

if complaints_csv.is_file() and population_csv.is_file():
    processed_rows = process_complaints_csv(
        complaints_csv,
        population_csv,
        enriched_csv,
        state_population_column="state",  # Change to Census header, e.g. NAME.
        population_column="population",  # Change to Census header, e.g. POPESTIMATE2024.
        chunksize=100_000,
    )
    print(f"Wrote {processed_rows:,} complaints to {enriched_csv}")
else:
    print("Add both the CFPB complaints CSV and user-provided population CSV, then rerun.")

### Why per-capita rates are fairer

A state with ten times the population can have many more complaints even if each resident is just as likely to complain. Dividing complaint volume by residents and scaling to 100,000 compares complaint intensity relative to population rather than state size. Use the rate alongside counts: a small state's rate may move sharply from a few complaints, and population alone is not the bank's customer base. For risk prioritization, also consider a bank-specific customer denominator, product mix, complaint severity, and data coverage.